# Oráculos y relativización

Este cuaderno ilustra la computación con oráculo y el teorema de Baker-Gill-Solovay.
Un **oráculo** es una caja negra que responde consultas en tiempo O(1).

Artículo: `03/11-oráculos-y-relativización.md`

In [ ]:
import random
from collections import defaultdict

## 1. Oráculo de SAT

Simulamos un oráculo para 3-SAT. Desde fuera es una caja negra: recibe una fórmula y devuelve si es satisfacible.

In [ ]:
def oraculo_sat(formula, n_vars):
    for bits in range(1 << n_vars):
        asig = {v+1: bool((bits >> v) & 1) for v in range(n_vars)}
        if all(any((l>0)==asig.get(abs(l), False) for l in c) for c in formula):
            return True, asig
    return False, None

f_sat = [[1, -2], [-1, 3], [2, -3]]
f_unsat = [[1, 2], [-1, 2], [1, -2], [-1, -2]]

sat1, asig = oraculo_sat(f_sat, 3)
sat2, _    = oraculo_sat(f_unsat, 2)
print(f"Formula 1 satisfacible: {sat1}, asignacion: {asig}")
print(f"Formula 2 satisfacible: {sat2}")

## 2. Problema en P^SAT: busqueda binaria sobre el numero de variables usadas

Con un oráculo SAT, podemos resolver en P^SAT el problema 'minima cantidad de variables necesaria'.

In [ ]:
def min_vars_necesarias(formula, n_vars):
    """Busqueda binaria: minima k tal que existe asig sobre k vars que satisface formula."""
    for k in range(1, n_vars+1):
        sat, _ = oraculo_sat(formula, k)
        if sat:
            return k
    return None

for nombre, f, n in [("f_sat", f_sat, 3), ("f_unsat", f_unsat, 2)]:
    k = min_vars_necesarias(f, n)
    print(f"{nombre}: min vars necesarias = {k}")

## 3. Baker-Gill-Solovay (1975): P^A = NP^A para ciertos oráculos A

Elegimos A = PARITY (par número de cadenas aceptadas). Con este oráculo, NP^A no parece más potente que P^A para este problema específico.

In [ ]:
def oraculo_paridad(longitud, semilla=42):
    """Oraculo A: subconjunto aleatorio de {0,1}^longitud."""
    rng = random.Random(semilla)
    return {format(i, f'0{longitud}b') for i in range(1 << longitud) if rng.random() < 0.5}

def problema_paridad_L_B(n, oraculo):
    """L_B: {n | |B ∩ {0,1}^n| es impar}. Este lenguaje separa P^B de NP^B."""
    count = sum(1 for s in oraculo if len(s) == n)
    return count % 2 == 1

print("Oraculo B aleatorio y lenguaje L_B:")
for longitud in range(1, 7):
    A = oraculo_paridad(longitud)
    en_LB = problema_paridad_L_B(longitud, A)
    total = 1 << longitud
    en_A = sum(1 for s in A if len(s) == longitud)
    print(f"  n={longitud}: {en_A}/{total} cadenas en B, |B∩{{0,1}}^{longitud}| {'impar' if en_LB else 'par'} -> {longitud} {'en' if en_LB else 'no en'} L_B")

## 4. Implicaciones para P vs. NP

El teorema de Baker-Gill-Solovay muestra que:
1. Existen oráculos A donde P^A = NP^A
2. Existen oráculos B donde P^B ≠ NP^B

Cualquier técnica de prueba que *relativice* no puede resolver P vs. NP.

In [ ]:
print("Resumen del teorema de Baker-Gill-Solovay (1975):")
print()
print("Existe oraculo A tal que P^A = NP^A")
print("  -> Tecnicas de colapso no pueden probar P != NP")
print()
print("Existe oraculo B tal que P^B != NP^B")
print("  -> Tecnicas de separacion pura no pueden probar P = NP")
print()
print("Consecuencia: la diagonalizacion clasica no puede resolver P vs. NP.")
print()
print("Barreras conocidas:")
print("  1. Relativizacion (Baker-Gill-Solovay 1975)")
print("  2. Natural proofs (Razborov-Rudich 1994)")
print("  3. Algebrizacion (Aaronson-Wigderson 2009)")
print()
print("Cualquier prueba de P vs. NP debe evitar las tres barreras.")